# SE-BRL Dataset Audit

This notebook performs the initial data-quality audit of the raw datasets used in the SE-BRL experiments.

The audit examines:

- dataset dimensions and schema
- data types
- missing values
- exact duplicate records
- unique values and cardinality
- class/label distributions
- empty textual fields
- basic text and URL statistics
- potential identifiers relevant to grouping and leakage prevention

Raw source files are treated as immutable. No records are modified, removed, relabeled, or transformed during this stage.

In [2]:
import sys
print(sys.executable)

/Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/.venv/bin/python


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import arff

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("Environment ready.")

pandas: 2.3.3
numpy: 2.0.2
Environment ready.


In [4]:
# Locate the repository root automatically
cwd = Path.cwd().resolve()

REPO_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / ".git").exists() and (p / "experiments").exists()
    ),
    None
)

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repository root.")

DATA_ROOT = REPO_ROOT / "experiments" / "datasets-raw"
AUDIT_DIR = REPO_ROOT / "experiments" / "results" / "audits"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Raw datasets:", DATA_ROOT)
print("Datasets folder exists:", DATA_ROOT.exists())
print("Audit output:", AUDIT_DIR)

Repository root: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL
Raw datasets: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/datasets-raw
Datasets folder exists: True
Audit output: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/audits


In [7]:
print("RAW DATASET FILES FOUND\n")

for path in sorted(DATA_ROOT.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"{path.relative_to(DATA_ROOT)}"
            f"  |  {size_mb:.2f} MB"
        )

RAW DATASET FILES FOUND

.DS_Store  |  0.01 MB
email/.DS_Store  |  0.01 MB
email/ceas_08/.DS_Store  |  0.01 MB
email/ceas_08/CEAS_08.csv  |  64.76 MB
email/enron/.DS_Store  |  0.01 MB
email/enron/Enron.csv  |  42.76 MB
email/nazario/.DS_Store  |  0.01 MB
email/nazario/Nazario.csv  |  7.45 MB
email/spamassasin/SpamAssasin.csv  |  14.19 MB
url/.DS_Store  |  0.01 MB
url/iscx-url-2016/.DS_Store  |  0.01 MB
url/iscx-url-2016/All.csv  |  11.12 MB
url/iscx-url-2016/Phishing.csv  |  4.42 MB
webpage/.DS_Store  |  0.01 MB
webpage/PhiUSIL/.DS_Store  |  0.01 MB
webpage/PhiUSIL/PhiUSIIL_Phishing_URL_Dataset.csv  |  54.22 MB
webpage/PhiUSIL/phiusiil+phishing+url+dataset.zip  |  14.66 MB
webpage/uci/.DS_Store  |  0.01 MB
webpage/uci/phishing+websites/.old.arff  |  0.16 MB
webpage/uci/phishing+websites/Phishing Websites Features.docx  |  0.04 MB
webpage/uci/phishing+websites/Training Dataset.arff  |  0.76 MB
webpage/uci/phishing+websites.zip  |  0.96 MB


## Dataset Manifest

The manifest identifies the raw source files used in the SE-BRL dataset audit.
Archive files, operating-system metadata, and supplementary documentation are
excluded from dataset-level analysis. Raw source files remain unmodified.

In [9]:
DATASETS = {
    "CEAS-08": {
        "modality": "email",
        "path": DATA_ROOT / "email" / "ceas_08" / "CEAS_08.csv",
        "format": "csv",
    },

    "Enron": {
        "modality": "email",
        "path": DATA_ROOT / "email" / "enron" / "Enron.csv",
        "format": "csv",
    },

    "Nazario": {
        "modality": "email",
        "path": DATA_ROOT / "email" / "nazario" / "Nazario.csv",
        "format": "csv",
    },

    "SpamAssassin": {
        "modality": "email",
        "path": DATA_ROOT / "email" / "spamassasin" / "SpamAssasin.csv",
        "format": "csv",
    },

    "PhiUSIIL": {
        "modality": "webpage",
        "path": DATA_ROOT / "webpage" / "PhiUSIL" / "PhiUSIIL_Phishing_URL_Dataset.csv",
        "format": "csv",
    },

    "UCI Phishing Websites": {
        "modality": "webpage",
        "path": DATA_ROOT / "webpage" / "uci" / "phishing+websites" / "Training Dataset.arff",
        "format": "arff",
    },

    "ISCX-URL2016": {
        "modality": "url",
        "paths": [
            DATA_ROOT / "url" / "iscx-url-2016" / "All.csv",
            DATA_ROOT / "url" / "iscx-url-2016" / "Phishing.csv",
        ],
        "format": "csv",
    },
}

In [10]:
for name, info in DATASETS.items():
    paths = info.get("paths", [info.get("path")])

    print(f"\n{name} [{info['modality']}]")

    for path in paths:
        print(f"  {path.relative_to(DATA_ROOT)}")
        print(f"  Exists: {path.exists()}")


CEAS-08 [email]
  email/ceas_08/CEAS_08.csv
  Exists: True

Enron [email]
  email/enron/Enron.csv
  Exists: True

Nazario [email]
  email/nazario/Nazario.csv
  Exists: True

SpamAssassin [email]
  email/spamassasin/SpamAssasin.csv
  Exists: True

PhiUSIIL [webpage]
  webpage/PhiUSIL/PhiUSIIL_Phishing_URL_Dataset.csv
  Exists: True

UCI Phishing Websites [webpage]
  webpage/uci/phishing+websites/Training Dataset.arff
  Exists: True

ISCX-URL2016 [url]
  url/iscx-url-2016/All.csv
  Exists: True
  url/iscx-url-2016/Phishing.csv
  Exists: True


## 1. Dataset Structure and Schema Audit

This section examines the basic structure of each raw dataset, including its
dimensions, column names, and data types. No records are modified during the
inspection.

In [17]:
def load_raw_dataset(path, file_format):
    """
    Load a raw dataset for inspection without modifying its contents.
    """
    if file_format == "csv":
        return pd.read_csv(path, low_memory=False)

    elif file_format == "arff":
        data, metadata = arff.loadarff(path)
        return pd.DataFrame(data)

    else:
        raise ValueError(f"Unsupported format: {file_format}")

In [16]:
loaded_datasets = {}

for name, info in DATASETS.items():

    # ISCX has multiple files and will be inspected separately
    if name == "ISCX-URL2016":
        continue

    df = load_raw_dataset(
        info["path"],
        info["format"]
    )

    loaded_datasets[name] = df

    print("=" * 80)
    print(f"DATASET: {name}")
    print(f"MODALITY: {info['modality']}")
    print(f"SHAPE: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    print("\nCOLUMNS:")

    for column in df.columns:
        print(f"  - {column}: {df[column].dtype}")

    print()

DATASET: CEAS-08
MODALITY: email
SHAPE: 39,154 rows × 7 columns

COLUMNS:
  - sender: object
  - receiver: object
  - date: object
  - subject: object
  - body: object
  - label: int64
  - urls: int64

DATASET: Enron
MODALITY: email
SHAPE: 29,767 rows × 3 columns

COLUMNS:
  - subject: object
  - body: object
  - label: int64

DATASET: Nazario
MODALITY: email
SHAPE: 1,565 rows × 7 columns

COLUMNS:
  - sender: object
  - receiver: object
  - date: object
  - subject: object
  - body: object
  - urls: int64
  - label: int64

DATASET: SpamAssassin
MODALITY: email
SHAPE: 5,809 rows × 7 columns

COLUMNS:
  - sender: object
  - receiver: object
  - date: object
  - subject: object
  - body: object
  - label: int64
  - urls: int64

DATASET: PhiUSIIL
MODALITY: webpage
SHAPE: 235,795 rows × 56 columns

COLUMNS:
  - FILENAME: object
  - URL: object
  - URLLength: int64
  - Domain: object
  - DomainLength: int64
  - IsDomainIP: int64
  - TLD: object
  - URLSimilarityIndex: float64
  - CharContin

In [18]:
iscx_info = DATASETS["ISCX-URL2016"]

iscx_datasets = {}

for path in iscx_info["paths"]:
    df = load_raw_dataset(path, iscx_info["format"])
    iscx_datasets[path.name] = df

    print("=" * 80)
    print(f"FILE: {path.name}")
    print(f"SHAPE: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    print(f"COLUMNS: {list(df.columns)}")
    print()

FILE: All.csv
SHAPE: 36,707 rows × 80 columns
COLUMNS: ['Querylength', 'domain_token_count', 'path_token_count', 'avgdomaintokenlen', 'longdomaintokenlen', 'avgpathtokenlen', 'tld', 'charcompvowels', 'charcompace', 'ldl_url', 'ldl_domain', 'ldl_path', 'ldl_filename', 'ldl_getArg', 'dld_url', 'dld_domain', 'dld_path', 'dld_filename', 'dld_getArg', 'urlLen', 'domainlength', 'pathLength', 'subDirLen', 'fileNameLen', 'this.fileExtLen', 'ArgLen', 'pathurlRatio', 'ArgUrlRatio', 'argDomanRatio', 'domainUrlRatio', 'pathDomainRatio', 'argPathRatio', 'executable', 'isPortEighty', 'NumberofDotsinURL', 'ISIpAddressInDomainName', 'CharacterContinuityRate', 'LongestVariableValue', 'URL_DigitCount', 'host_DigitCount', 'Directory_DigitCount', 'File_name_DigitCount', 'Extension_DigitCount', 'Query_DigitCount', 'URL_Letter_Count', 'host_letter_count', 'Directory_LetterCount', 'Filename_LetterCount', 'Extension_LetterCount', 'Query_LetterCount', 'LongestPathTokenLength', 'Domain_LongestWordLength', 'Path

## 2. Native Label and Class Distribution Audit

This section examines the label fields provided by each source dataset and
their observed class distributions. Labels are reported in their original
form. No label values are manually assigned, remapped, or modified during
the audit.

In [19]:
label_columns = {
    "CEAS-08": "label",
    "Enron": "label",
    "Nazario": "label",
    "SpamAssassin": "label",
    "PhiUSIIL": "label",
    "UCI Phishing Websites": "Result",
}

for name, label_col in label_columns.items():
    df = loaded_datasets[name]

    print("=" * 80)
    print(f"DATASET: {name}")
    print(f"LABEL COLUMN: {label_col}")

    print("\nCLASS COUNTS:")
    print(df[label_col].value_counts(dropna=False))

    print("\nCLASS PERCENTAGES:")
    print(
        df[label_col]
        .value_counts(dropna=False, normalize=True)
        .mul(100)
        .round(2)
    )

    print()

DATASET: CEAS-08
LABEL COLUMN: label

CLASS COUNTS:
label
1    21842
0    17312
Name: count, dtype: int64

CLASS PERCENTAGES:
label
1    55.78
0    44.22
Name: proportion, dtype: float64

DATASET: Enron
LABEL COLUMN: label

CLASS COUNTS:
label
0    15791
1    13976
Name: count, dtype: int64

CLASS PERCENTAGES:
label
0    53.05
1    46.95
Name: proportion, dtype: float64

DATASET: Nazario
LABEL COLUMN: label

CLASS COUNTS:
label
1    1565
Name: count, dtype: int64

CLASS PERCENTAGES:
label
1    100.0
Name: proportion, dtype: float64

DATASET: SpamAssassin
LABEL COLUMN: label

CLASS COUNTS:
label
0    4091
1    1718
Name: count, dtype: int64

CLASS PERCENTAGES:
label
0    70.43
1    29.57
Name: proportion, dtype: float64

DATASET: PhiUSIIL
LABEL COLUMN: label

CLASS COUNTS:
label
1    134850
0    100945
Name: count, dtype: int64

CLASS PERCENTAGES:
label
1    57.19
0    42.81
Name: proportion, dtype: float64

DATASET: UCI Phishing Websites
LABEL COLUMN: Result

CLASS COUNTS:
Result
b'1' 

In [20]:
for filename, df in iscx_datasets.items():
    print("=" * 80)
    print(f"ISCX FILE: {filename}")
    print(f"SHAPE: {df.shape}")

    possible_labels = [
        col for col in df.columns
        if any(
            keyword in col.lower()
            for keyword in ["label", "class", "type", "category"]
        )
    ]

    print("Possible label columns:", possible_labels)

    for col in possible_labels:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False).head(20))

    print()

ISCX FILE: All.csv
SHAPE: (36707, 80)
Possible label columns: ['URL_Type_obf_Type']

URL_Type_obf_Type:
URL_Type_obf_Type
Defacement    7930
benign        7781
phishing      7586
malware       6712
spam          6698
Name: count, dtype: int64

ISCX FILE: Phishing.csv
SHAPE: (15367, 80)
Possible label columns: ['URL_Type_obf_Type']

URL_Type_obf_Type:
URL_Type_obf_Type
benign      7781
phishing    7586
Name: count, dtype: int64



In [ ]:
## 3. Missing-Value Audit

This section measures missing values in each raw dataset. Missingness is
reported without imputing, removing, or otherwise modifying any records.

In [21]:
def audit_missing_values(df, dataset_name):
    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2)

    report = pd.DataFrame({
        "dataset": dataset_name,
        "column": df.columns,
        "missing_count": missing_count.values,
        "missing_percent": missing_pct.values
    })

    return report


missing_reports = []

# Main six datasets
for name, df in loaded_datasets.items():
    missing_reports.append(
        audit_missing_values(df, name)
    )

# ISCX files
for filename, df in iscx_datasets.items():
    missing_reports.append(
        audit_missing_values(
            df,
            f"ISCX-URL2016/{filename}"
        )
    )

missing_report = pd.concat(
    missing_reports,
    ignore_index=True
)

In [22]:
missing_summary = missing_report[
    missing_report["missing_count"] > 0
].sort_values(
    ["dataset", "missing_percent"],
    ascending=[True, False]
)

if missing_summary.empty:
    print("No missing values detected.")
else:
    display(missing_summary)

,dataset,column,missing_count,missing_percent
1,CEAS-08,receiver,462,1.18
3,CEAS-08,subject,28,0.07
7,Enron,subject,198,0.67
176,ISCX-URL2016/All.csv,NumberRate_Extension,10130,27.60
186,ISCX-URL2016/All.csv,Entropy_DirectoryName,8468,23.07
116,ISCX-URL2016/All.csv,avgpathtokenlen,280,0.76
187,ISCX-URL2016/All.csv,Entropy_Filename,236,0.64
188,ISCX-URL2016/All.csv,Entropy_Extension,40,0.11
174,ISCX-URL2016/All.csv,NumberRate_DirectoryName,10,0.03
175,ISCX-URL2016/All.csv,NumberRate_FileName,10,0.03


In [23]:
dataset_missing_summary = (
    missing_report
    .groupby("dataset")
    .agg(
        columns_audited=("column", "count"),
        columns_with_missing=("missing_count", lambda x: (x > 0).sum()),
        total_missing_values=("missing_count", "sum"),
        highest_missing_percent=("missing_percent", "max")
    )
    .reset_index()
)

display(dataset_missing_summary)

,dataset,columns_audited,columns_with_missing,total_missing_values,highest_missing_percent
0,CEAS-08,7,2,490,1.18
1,Enron,3,1,198,0.67
2,ISCX-URL2016/All.csv,80,9,19183,27.60
3,ISCX-URL2016/Phishing.csv,80,9,9669,47.86
4,Nazario,7,3,101,6.13
5,PhiUSIIL,56,0,0,0.00
6,SpamAssassin,7,3,227,3.62
7,UCI Phishing Websites,31,0,0,0.00


## 4. Exact Duplicate Audit

This section identifies records that are exact duplicates within each raw
dataset. Duplicate records are measured only and are not removed during the
audit stage.

In [24]:
duplicate_results = []

# Main datasets
for name, df in loaded_datasets.items():
    duplicate_count = df.duplicated().sum()
    duplicate_percent = round(
        duplicate_count / len(df) * 100, 2
    )

    duplicate_results.append({
        "dataset": name,
        "rows": len(df),
        "exact_duplicate_rows": duplicate_count,
        "duplicate_percent": duplicate_percent
    })

# ISCX files
for filename, df in iscx_datasets.items():
    duplicate_count = df.duplicated().sum()
    duplicate_percent = round(
        duplicate_count / len(df) * 100, 2
    )

    duplicate_results.append({
        "dataset": f"ISCX-URL2016/{filename}",
        "rows": len(df),
        "exact_duplicate_rows": duplicate_count,
        "duplicate_percent": duplicate_percent
    })

duplicate_report = pd.DataFrame(duplicate_results)

display(
    duplicate_report.sort_values(
        "duplicate_percent",
        ascending=False
    )
)

,dataset,rows,exact_duplicate_rows,duplicate_percent
5,UCI Phishing Websites,11055,5206,47.09
6,ISCX-URL2016/All.csv,36707,9754,26.57
7,ISCX-URL2016/Phishing.csv,15367,544,3.54
0,CEAS-08,39154,0,0.00
1,Enron,29767,0,0.00
2,Nazario,1565,0,0.00
3,SpamAssassin,5809,0,0.00
4,PhiUSIIL,235795,0,0.00


## 5. Content-Level Duplicate Audit

This section examines duplicate observations using modality-relevant content
identifiers rather than requiring the entire record to be identical. This
helps identify repeated email content and repeated URLs that may present
data leakage risks during later dataset partitioning.

In [25]:
email_names = [
    "CEAS-08",
    "Enron",
    "Nazario",
    "SpamAssassin"
]

email_duplicate_results = []

for name in email_names:
    df = loaded_datasets[name]

    subject_dupes = df.duplicated(
        subset=["subject"],
        keep=False
    ).sum()

    body_dupes = df.duplicated(
        subset=["body"],
        keep=False
    ).sum()

    subject_body_dupes = df.duplicated(
        subset=["subject", "body"],
        keep=False
    ).sum()

    email_duplicate_results.append({
        "dataset": name,
        "rows": len(df),
        "rows_in_duplicate_subject_groups": subject_dupes,
        "rows_in_duplicate_body_groups": body_dupes,
        "rows_in_duplicate_subject_body_groups": subject_body_dupes
    })

email_content_duplicates = pd.DataFrame(
    email_duplicate_results
)

display(email_content_duplicates)

,dataset,rows,rows_in_duplicate_subject_groups,rows_in_duplicate_body_groups,rows_in_duplicate_subject_body_groups
0,CEAS-08,39154,27075,0,0
1,Enron,29767,8882,0,0
2,Nazario,1565,241,0,0
3,SpamAssassin,5809,2151,0,0


In [26]:
phiusiil = loaded_datasets["PhiUSIIL"]

phiusiil_url_audit = pd.DataFrame({
    "metric": [
        "Total rows",
        "Unique URLs",
        "Rows in duplicated URL groups"
    ],
    "value": [
        len(phiusiil),
        phiusiil["URL"].nunique(dropna=True),
        phiusiil.duplicated(
            subset=["URL"],
            keep=False
        ).sum()
    ]
})

display(phiusiil_url_audit)

,metric,value
0,Total rows,235795
1,Unique URLs,235370
2,Rows in duplicated URL groups,850


In [27]:
for filename, df in iscx_datasets.items():
    print("=" * 80)
    print(filename)

    url_candidates = [
        col for col in df.columns
        if "url" in col.lower()
    ]

    print("URL-related columns:")
    for col in url_candidates:
        print(f"  - {col}")

    print()

All.csv
URL-related columns:
  - ldl_url
  - dld_url
  - urlLen
  - pathurlRatio
  - ArgUrlRatio
  - domainUrlRatio
  - NumberofDotsinURL
  - URL_DigitCount
  - URL_Letter_Count
  - URL_sensitiveWord
  - URLQueries_variable
  - spcharUrl
  - NumberRate_URL
  - SymbolCount_URL
  - Entropy_URL
  - URL_Type_obf_Type

Phishing.csv
URL-related columns:
  - ldl_url
  - dld_url
  - urlLen
  - pathurlRatio
  - ArgUrlRatio
  - domainUrlRatio
  - NumberofDotsinURL
  - URL_DigitCount
  - URL_Letter_Count
  - URL_sensitiveWord
  - URLQueries_variable
  - spcharUrl
  - NumberRate_URL
  - SymbolCount_URL
  - Entropy_URL
  - URL_Type_obf_Type



In [28]:
for filename, df in iscx_datasets.items():
    print("=" * 80)
    print(filename)

    url_candidates = [
        col for col in df.columns
        if "url" in col.lower()
    ]

    for col in url_candidates:
        print(
            f"{col}: "
            f"{df[col].nunique(dropna=True):,} unique / "
            f"{len(df):,} rows"
        )

    print()

All.csv
ldl_url: 133 unique / 36,707 rows
dld_url: 34 unique / 36,707 rows
urlLen: 389 unique / 36,707 rows
pathurlRatio: 2,802 unique / 36,707 rows
ArgUrlRatio: 2,772 unique / 36,707 rows
domainUrlRatio: 2,519 unique / 36,707 rows
NumberofDotsinURL: 20 unique / 36,707 rows
URL_DigitCount: 186 unique / 36,707 rows
URL_Letter_Count: 483 unique / 36,707 rows
URL_sensitiveWord: 4 unique / 36,707 rows
URLQueries_variable: 15 unique / 36,707 rows
spcharUrl: 17 unique / 36,707 rows
NumberRate_URL: 3,575 unique / 36,707 rows
SymbolCount_URL: 40 unique / 36,707 rows
Entropy_URL: 23,450 unique / 36,707 rows
URL_Type_obf_Type: 5 unique / 36,707 rows

Phishing.csv
ldl_url: 47 unique / 15,367 rows
dld_url: 33 unique / 15,367 rows
urlLen: 228 unique / 15,367 rows
pathurlRatio: 2,047 unique / 15,367 rows
ArgUrlRatio: 1,230 unique / 15,367 rows
domainUrlRatio: 1,936 unique / 15,367 rows
NumberofDotsinURL: 20 unique / 15,367 rows
URL_DigitCount: 106 unique / 15,367 rows
URL_Letter_Count: 185 unique / 

In [29]:
for filename, df in iscx_datasets.items():
    print("=" * 80)
    print(filename)

    print(
        df["URL_Type_obf_Type"]
        .value_counts(dropna=False)
    )

    print()

All.csv
URL_Type_obf_Type
Defacement    7930
benign        7781
phishing      7586
malware       6712
spam          6698
Name: count, dtype: int64

Phishing.csv
URL_Type_obf_Type
benign      7781
phishing    7586
Name: count, dtype: int64



In [30]:
all_df = iscx_datasets["All.csv"]
phishing_df = iscx_datasets["Phishing.csv"]

common_columns = list(
    set(all_df.columns).intersection(phishing_df.columns)
)

print("All.csv columns:", len(all_df.columns))
print("Phishing.csv columns:", len(phishing_df.columns))
print("Common columns:", len(common_columns))

# Compare unique complete rows using the common schema
all_unique = all_df[common_columns].drop_duplicates()
phishing_unique = phishing_df[common_columns].drop_duplicates()

overlap = phishing_unique.merge(
    all_unique,
    on=common_columns,
    how="inner"
)

print("\nUnique rows in All.csv:", len(all_unique))
print("Unique rows in Phishing.csv:", len(phishing_unique))
print("Unique Phishing.csv rows also found in All.csv:", len(overlap))

if len(phishing_unique) > 0:
    print(
        "Overlap percentage:",
        round(len(overlap) / len(phishing_unique) * 100, 2),
        "%"
    )

All.csv columns: 80
Phishing.csv columns: 80
Common columns: 80

Unique rows in All.csv: 26953
Unique rows in Phishing.csv: 14823
Unique Phishing.csv rows also found in All.csv: 14823
Overlap percentage: 100.0 %


### ISCX-URL2016 File Relationship

The ISCX-URL2016 `All.csv` file contains five native classes: Defacement,
benign, phishing, malware, and spam. In contrast, `Phishing.csv` contains
only the benign and phishing classes.

Both files contain the same 80 columns. After removing exact duplicate rows,
all 14,823 unique feature vectors observed in `Phishing.csv` were also found
in `All.csv`, corresponding to 100% feature-vector overlap.

Therefore, `Phishing.csv` is treated as a benign/phishing subset of
`All.csv` rather than as an independent additional source. The two files
must not be concatenated, as doing so would duplicate observations.

The available files do not contain a raw URL identifier; therefore,
URL-level identity and duplicate analysis cannot be performed for
ISCX-URL2016. The reported overlap refers to complete feature-vector
equality.

## 6. Content Validity Audit

This section examines modality-relevant fields for empty or structurally
invalid values. The audit distinguishes missing values from values that are
present but contain no meaningful content. No records are modified.

In [31]:
email_validity_results = []

for name in [
    "CEAS-08",
    "Enron",
    "Nazario",
    "SpamAssassin"
]:
    df = loaded_datasets[name]

    empty_subject = (
        df["subject"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    empty_body = (
        df["body"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    empty_both = (
        df["subject"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        &
        df["body"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    ).sum()

    email_validity_results.append({
        "dataset": name,
        "rows": len(df),
        "empty_or_missing_subject": empty_subject,
        "empty_or_missing_body": empty_body,
        "empty_or_missing_both": empty_both
    })

email_validity_report = pd.DataFrame(
    email_validity_results
)

display(email_validity_report)

,dataset,rows,empty_or_missing_subject,empty_or_missing_body,empty_or_missing_both
0,CEAS-08,39154,28,0,0
1,Enron,29767,198,0,0
2,Nazario,1565,4,2,0
3,SpamAssassin,5809,16,2,0


### PhiUSIIL URL Validity

The raw URL field in PhiUSIIL is examined for missing, blank, and
structurally parseable values. This audit does not modify or normalize
the URLs.

In [32]:
from urllib.parse import urlparse

phiusiil = loaded_datasets["PhiUSIIL"]

urls = phiusiil["URL"]

missing_urls = urls.isna().sum()

blank_urls = (
    urls.fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

def has_parseable_host(value):
    if pd.isna(value):
        return False

    value = str(value).strip()

    if not value:
        return False

    try:
        parsed = urlparse(value)

        # Some URLs may omit the scheme
        if not parsed.netloc:
            parsed = urlparse("//" + value)

        return bool(parsed.netloc)

    except Exception:
        return False


parseable_mask = urls.apply(has_parseable_host)

phiusiil_url_validity = pd.DataFrame({
    "metric": [
        "Total rows",
        "Missing URLs",
        "Blank URLs",
        "Parseable host",
        "No parseable host",
        "Unique URLs"
    ],
    "value": [
        len(phiusiil),
        missing_urls,
        blank_urls,
        parseable_mask.sum(),
        (~parseable_mask).sum(),
        urls.nunique(dropna=True)
    ]
})

display(phiusiil_url_validity)

,metric,value
0,Total rows,235795
1,Missing URLs,0
2,Blank URLs,0
3,Parseable host,235795
4,No parseable host,0
5,Unique URLs,235370


## 7. Cross-Source Overlap and Leakage Audit

This section examines whether identical email content occurs across different
source datasets. Cross-source overlap is important because duplicated content
across sources could introduce information leakage if equivalent observations
are assigned to different development, calibration, or test partitions.

Comparisons are performed automatically using normalized text representations
for identity detection only. No labels or records are modified.

In [33]:
import hashlib
import re

def normalize_text_for_identity(value):
    """
    Conservative normalization used only for duplicate identity checking.
    This does not modify the raw dataset.
    """
    if pd.isna(value):
        return ""

    value = str(value)
    value = value.strip()
    value = re.sub(r"\s+", " ", value)

    return value


def text_hash(value):
    normalized = normalize_text_for_identity(value)
    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()


email_identity = {}

for name in [
    "CEAS-08",
    "Enron",
    "Nazario",
    "SpamAssassin"
]:
    df = loaded_datasets[name]

    temp = pd.DataFrame()

    temp["body_hash"] = (
        df["body"]
        .apply(text_hash)
    )

    temp["subject_body_hash"] = (
        (
            df["subject"]
            .apply(normalize_text_for_identity)
            + "\n"
            + df["body"]
            .apply(normalize_text_for_identity)
        )
        .apply(text_hash)
    )

    email_identity[name] = temp

In [34]:
from itertools import combinations

cross_source_results = []

for dataset_a, dataset_b in combinations(
    email_identity.keys(), 2
):
    a = email_identity[dataset_a]
    b = email_identity[dataset_b]

    body_overlap = len(
        set(a["body_hash"])
        &
        set(b["body_hash"])
    )

    subject_body_overlap = len(
        set(a["subject_body_hash"])
        &
        set(b["subject_body_hash"])
    )

    cross_source_results.append({
        "dataset_a": dataset_a,
        "dataset_b": dataset_b,
        "shared_unique_bodies": body_overlap,
        "shared_unique_subject_body": subject_body_overlap
    })


cross_source_email_overlap = pd.DataFrame(
    cross_source_results
)

display(cross_source_email_overlap)

,dataset_a,dataset_b,shared_unique_bodies,shared_unique_subject_body
0,CEAS-08,Enron,0,0
1,CEAS-08,Nazario,0,0
2,CEAS-08,SpamAssassin,0,0
3,Enron,Nazario,2,0
4,Enron,SpamAssassin,0,0
5,Nazario,SpamAssassin,1,0


### Cross-Source Email Overlap Finding

Cross-source comparison identified minimal overlap among the four email
datasets. Two normalized email bodies were shared between Enron and Nazario,
and one normalized body was shared between Nazario and SpamAssassin.
No shared normalized bodies were identified for the remaining source pairs.

Importantly, no identical subject-and-body combinations were observed across
any pair of email datasets. These findings indicate limited cross-source
content overlap, although the identified shared bodies should still be
considered during later grouping and partitioning to prevent potential
information leakage.

## 8. Content Statistics

This section summarizes basic characteristics of the textual content available
in the email datasets. Length statistics are measured without modifying the
underlying raw records.

In [35]:
email_content_stats = []

for name in [
    "CEAS-08",
    "Enron",
    "Nazario",
    "SpamAssassin"
]:
    df = loaded_datasets[name]

    subject_length = (
        df["subject"]
        .fillna("")
        .astype(str)
        .str.len()
    )

    body_length = (
        df["body"]
        .fillna("")
        .astype(str)
        .str.len()
    )

    combined_length = subject_length + body_length

    email_content_stats.append({
        "dataset": name,
        "rows": len(df),

        "subject_median_chars": round(subject_length.median(), 1),
        "subject_mean_chars": round(subject_length.mean(), 1),
        "subject_max_chars": subject_length.max(),

        "body_median_chars": round(body_length.median(), 1),
        "body_mean_chars": round(body_length.mean(), 1),
        "body_max_chars": body_length.max(),

        "combined_median_chars": round(combined_length.median(), 1),
        "combined_mean_chars": round(combined_length.mean(), 1),
        "combined_max_chars": combined_length.max()
    })

email_content_statistics = pd.DataFrame(
    email_content_stats
)

display(email_content_statistics)

,dataset,rows,subject_median_chars,subject_mean_chars,subject_max_chars,body_median_chars,body_mean_chars,body_max_chars,combined_median_chars,combined_mean_chars,combined_max_chars
0,CEAS-08,39154,35.0,38.9,285,570.0,1571.1,143996,602.0,1610.0,144050
1,Enron,29767,30.0,34.1,7170,694.0,1464.6,228353,728.0,1498.7,228367
2,Nazario,1565,36.0,40.6,510,720.0,4730.0,4599644,765.0,4770.6,4599693
3,SpamAssassin,5809,35.0,37.5,242,1084.0,2406.5,299201,1122.0,2444.0,299271


### Email Length Distribution and Extreme Values

Because maximum email lengths substantially exceed typical observations,
additional percentile statistics are examined to identify extreme-length
records. These records are reported only and are not removed or truncated
during the audit.

In [36]:
email_length_distribution = []

for name in [
    "CEAS-08",
    "Enron",
    "Nazario",
    "SpamAssassin"
]:
    df = loaded_datasets[name]

    body_length = (
        df["body"]
        .fillna("")
        .astype(str)
        .str.len()
    )

    email_length_distribution.append({
        "dataset": name,
        "p50": round(body_length.quantile(0.50), 1),
        "p90": round(body_length.quantile(0.90), 1),
        "p95": round(body_length.quantile(0.95), 1),
        "p99": round(body_length.quantile(0.99), 1),
        "p99_9": round(body_length.quantile(0.999), 1),
        "max": body_length.max(),
        "over_100k_chars": (body_length > 100_000).sum(),
        "over_1m_chars": (body_length > 1_000_000).sum()
    })

email_length_distribution = pd.DataFrame(
    email_length_distribution
)

display(email_length_distribution)

,dataset,p50,p90,p95,p99,p99_9,max,over_100k_chars,over_1m_chars
0,CEAS-08,570.0,3898.0,4749.0,13260.2,46441.0,143996,2,0
1,Enron,694.0,3081.4,4573.7,9390.7,69560.0,228353,10,0
2,Nazario,720.0,2332.0,3428.6,7868.4,456935.5,4599644,2,2
3,SpamAssassin,1084.0,4236.6,8120.8,20039.9,92965.2,299201,6,0


## 9. Label Consistency and Class Distribution

Native dataset labels are examined for class membership, frequency, and
proportion. Labels remain in their original source representation during
the audit. No manual relabeling or class mapping is performed.

In [37]:
label_audit_results = []

# Main datasets
label_columns = {
    "CEAS-08": "label",
    "Enron": "label",
    "Nazario": "label",
    "SpamAssassin": "label",
    "PhiUSIIL": "label",
    "UCI Phishing Websites": "Result",
}

for name, label_col in label_columns.items():
    df = loaded_datasets[name]

    counts = df[label_col].value_counts(
        dropna=False
    )

    for label_value, count in counts.items():
        label_audit_results.append({
            "dataset": name,
            "label_column": label_col,
            "native_label": repr(label_value),
            "count": count,
            "percent": round(
                count / len(df) * 100,
                2
            )
        })


# ISCX All.csv only
# Phishing.csv is already established as a subset of All.csv.
iscx_all = iscx_datasets["All.csv"]

counts = iscx_all[
    "URL_Type_obf_Type"
].value_counts(dropna=False)

for label_value, count in counts.items():
    label_audit_results.append({
        "dataset": "ISCX-URL2016",
        "label_column": "URL_Type_obf_Type",
        "native_label": repr(label_value),
        "count": count,
        "percent": round(
            count / len(iscx_all) * 100,
            2
        )
    })


label_audit = pd.DataFrame(
    label_audit_results
)

display(label_audit)

,dataset,label_column,native_label,count,percent
0,CEAS-08,label,1,21842,55.78
1,CEAS-08,label,0,17312,44.22
2,Enron,label,0,15791,53.05
3,Enron,label,1,13976,46.95
4,Nazario,label,1,1565,100.00
5,SpamAssassin,label,0,4091,70.43
6,SpamAssassin,label,1,1718,29.57
7,PhiUSIIL,label,1,134850,57.19
8,PhiUSIIL,label,0,100945,42.81
9,UCI Phishing Websites,Result,b'1',6157,55.69


In [38]:
verified_label_semantics = pd.DataFrame([
    {
        "dataset": "PhiUSIIL",
        "native_label": "1",
        "verified_meaning": "legitimate",
        "status": "verified from source documentation"
    },
    {
        "dataset": "PhiUSIIL",
        "native_label": "0",
        "verified_meaning": "phishing",
        "status": "verified from source documentation"
    },
    {
        "dataset": "UCI Phishing Websites",
        "native_label": "1",
        "verified_meaning": "legitimate",
        "status": "verified from source documentation"
    },
    {
        "dataset": "UCI Phishing Websites",
        "native_label": "-1",
        "verified_meaning": "phishy",
        "status": "verified from source documentation"
    },
    {
        "dataset": "ISCX-URL2016",
        "native_label": "native strings",
        "verified_meaning": "benign / phishing / spam / malware / Defacement",
        "status": "explicit in raw dataset"
    },
    {
        "dataset": "CEAS-08",
        "native_label": "0 / 1",
        "verified_meaning": "pending provenance verification",
        "status": "do not map yet"
    },
    {
        "dataset": "Enron",
        "native_label": "0 / 1",
        "verified_meaning": "pending provenance verification",
        "status": "do not map yet"
    },
    {
        "dataset": "Nazario",
        "native_label": "1",
        "verified_meaning": "pending provenance verification",
        "status": "do not map yet"
    },
    {
        "dataset": "SpamAssassin",
        "native_label": "0 / 1",
        "verified_meaning": "pending provenance verification",
        "status": "do not map yet"
    }
])

display(verified_label_semantics)

,dataset,native_label,verified_meaning,status
0,PhiUSIIL,1,legitimate,verified from source documentation
1,PhiUSIIL,0,phishing,verified from source documentation
2,UCI Phishing Websites,1,legitimate,verified from source documentation
3,UCI Phishing Websites,-1,phishy,verified from source documentation
4,ISCX-URL2016,native strings,benign / phishing / spam / malware / Defacement,explicit in raw dataset
5,CEAS-08,0 / 1,pending provenance verification,do not map yet
6,Enron,0 / 1,pending provenance verification,do not map yet
7,Nazario,1,pending provenance verification,do not map yet
8,SpamAssassin,0 / 1,pending provenance verification,do not map yet


## 10. Numeric Feature Sanity Audit

Numeric features in the structured webpage and URL datasets are examined for
missing values, infinite values, constant features, and observed value ranges.

This stage is descriptive only. No clipping, imputation, normalization,
feature removal, or other transformation is performed. Findings requiring
remediation are carried forward to the cleaning and preprocessing stages.

In [39]:
def audit_numeric_features(df, dataset_name):
    results = []

    numeric_columns = df.select_dtypes(
        include=[np.number]
    ).columns

    for column in numeric_columns:
        series = df[column]

        missing_count = series.isna().sum()

        finite_series = series.dropna()

        if len(finite_series) > 0:
            infinite_count = np.isinf(
                finite_series
            ).sum()

            finite_only = finite_series[
                np.isfinite(finite_series)
            ]
        else:
            infinite_count = 0
            finite_only = finite_series

        if len(finite_only) > 0:
            minimum = finite_only.min()
            maximum = finite_only.max()
        else:
            minimum = np.nan
            maximum = np.nan

        results.append({
            "dataset": dataset_name,
            "feature": column,
            "dtype": str(series.dtype),
            "missing": missing_count,
            "infinite": infinite_count,
            "unique_values": series.nunique(
                dropna=True
            ),
            "min": minimum,
            "max": maximum,
            "constant_feature": (
                series.nunique(dropna=True) <= 1
            )
        })

    return pd.DataFrame(results)

In [40]:
phiusiil_numeric_audit = audit_numeric_features(
    loaded_datasets["PhiUSIIL"],
    "PhiUSIIL"
)

iscx_numeric_audit = audit_numeric_features(
    iscx_datasets["All.csv"],
    "ISCX-URL2016"
)

structured_numeric_audit = pd.concat(
    [
        phiusiil_numeric_audit,
        iscx_numeric_audit
    ],
    ignore_index=True
)

print(
    "Numeric features audited:",
    len(structured_numeric_audit)
)

display(
    structured_numeric_audit[
        (structured_numeric_audit["missing"] > 0)
        |
        (structured_numeric_audit["infinite"] > 0)
        |
        (structured_numeric_audit["constant_feature"])
    ]
)

Numeric features audited: 130


,dataset,feature,dtype,missing,infinite,unique_values,min,max,constant_feature
56,ISCX-URL2016,avgpathtokenlen,float64,280,0,835,0.0,105.000000,False
82,ISCX-URL2016,argPathRatio,float64,0,10,2047,0.0,2.000000,False
86,ISCX-URL2016,ISIpAddressInDomainName,int64,0,0,1,-1.0,-1.000000,True
114,ISCX-URL2016,NumberRate_DirectoryName,float64,10,0,729,-1.0,0.935065,False
115,ISCX-URL2016,NumberRate_FileName,float64,10,0,2613,-1.0,1.000000,False
116,ISCX-URL2016,NumberRate_Extension,float64,10130,0,2032,-1.0,1.000000,False
117,ISCX-URL2016,NumberRate_AfterPath,float64,3,0,1851,-1.0,1.000000,False
126,ISCX-URL2016,Entropy_DirectoryName,float64,8468,0,3936,-1.0,0.962479,False
127,ISCX-URL2016,Entropy_Filename,float64,236,0,11513,-1.0,1.000000,False
128,ISCX-URL2016,Entropy_Extension,float64,40,0,6449,-1.0,1.000000,False


### UCI Phishing Websites Native Feature Values

The UCI Phishing Websites ARFF file is inspected in its native loaded
representation. Because nominal ARFF values are represented as byte objects,
no numeric conversion is performed during the audit.

The analysis records the number and set of unique native values for each
feature to identify unexpected categories or invariant features.

In [41]:
uci_df = loaded_datasets["UCI Phishing Websites"]

uci_native_value_audit = []

for column in uci_df.columns:
    unique_values = uci_df[column].dropna().unique()

    # Sort their string representations only for readable reporting.
    displayed_values = sorted(
        [repr(value) for value in unique_values]
    )

    uci_native_value_audit.append({
        "feature": column,
        "missing": uci_df[column].isna().sum(),
        "unique_count": len(unique_values),
        "native_values": ", ".join(displayed_values),
        "constant_feature": len(unique_values) <= 1
    })

uci_native_value_audit = pd.DataFrame(
    uci_native_value_audit
)

display(uci_native_value_audit)

,feature,missing,unique_count,native_values,constant_feature
0,having_IP_Address,0,2,"b'-1', b'1'",False
1,URL_Length,0,3,"b'-1', b'0', b'1'",False
2,Shortining_Service,0,2,"b'-1', b'1'",False
3,having_At_Symbol,0,2,"b'-1', b'1'",False
4,double_slash_redirecting,0,2,"b'-1', b'1'",False
5,Prefix_Suffix,0,2,"b'-1', b'1'",False
6,having_Sub_Domain,0,3,"b'-1', b'0', b'1'",False
7,SSLfinal_State,0,3,"b'-1', b'0', b'1'",False
8,Domain_registeration_length,0,2,"b'-1', b'1'",False
9,Favicon,0,2,"b'-1', b'1'",False


## 11. Grouping and Leakage Candidate Audit

Potential grouping variables are examined to support leakage-aware dataset
partitioning in later experimental stages.

The purpose of this audit is to identify repeated entities or content that
should not be allowed to appear independently across development,
calibration, and test partitions.

No grouping, deduplication, or partitioning is performed in this notebook.
Candidate grouping strategies are documented for subsequent preprocessing
and experimental design.

In [42]:
phiusiil_df = loaded_datasets["PhiUSIIL"]

domain_counts = (
    phiusiil_df["Domain"]
    .fillna("")
    .astype(str)
    .value_counts()
)

phiusiil_grouping_summary = pd.DataFrame([
    {
        "dataset": "PhiUSIIL",
        "candidate_group": "Domain",
        "total_rows": len(phiusiil_df),
        "unique_groups": phiusiil_df["Domain"].nunique(
            dropna=True
        ),
        "rows_in_repeated_groups": int(
            domain_counts[
                domain_counts > 1
            ].sum()
        ),
        "repeated_groups": int(
            (domain_counts > 1).sum()
        ),
        "largest_group_size": int(
            domain_counts.max()
        )
    }
])

display(phiusiil_grouping_summary)

,dataset,candidate_group,total_rows,unique_groups,rows_in_repeated_groups,repeated_groups,largest_group_size
0,PhiUSIIL,Domain,235795,220086,21235,5526,1197


In [43]:
domain_label_counts = (
    phiusiil_df
    .groupby("Domain")["label"]
    .nunique()
)

conflicting_domains = domain_label_counts[
    domain_label_counts > 1
]

phiusiil_domain_label_summary = pd.DataFrame([
    {
        "total_unique_domains": phiusiil_df[
            "Domain"
        ].nunique(dropna=True),

        "domains_with_multiple_labels":
            len(conflicting_domains),

        "percent_conflicting_domains": round(
            len(conflicting_domains)
            / phiusiil_df["Domain"].nunique(
                dropna=True
            )
            * 100,
            4
        )
    }
])

display(phiusiil_domain_label_summary)

,total_unique_domains,domains_with_multiple_labels,percent_conflicting_domains
0,220086,54,0.0245


### Email Grouping Candidates

Email datasets are examined for metadata that may support leakage-aware
grouping. Sender identity is considered a potential grouping variable where
available, while repeated subject lines alone are not treated as reliable
message identities.

Content hashes identified earlier may additionally be used to prevent
identical email bodies from crossing partition boundaries.

This section identifies candidate grouping information only. No groups or
partitions are created during the audit.

In [44]:
email_grouping_results = []

for name in [
    "CEAS-08",
    "Enron",
    "Nazario",
    "SpamAssassin"
]:
    df = loaded_datasets[name]

    if "sender" in df.columns:
        sender = (
            df["sender"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        valid_sender = sender[
            sender != ""
        ]

        sender_counts = valid_sender.value_counts()

        email_grouping_results.append({
            "dataset": name,
            "sender_available": True,
            "rows": len(df),
            "rows_with_sender": len(valid_sender),
            "unique_senders": valid_sender.nunique(),
            "repeated_senders": int(
                (sender_counts > 1).sum()
            ),
            "rows_in_repeated_sender_groups": int(
                sender_counts[
                    sender_counts > 1
                ].sum()
            ),
            "largest_sender_group": (
                int(sender_counts.max())
                if len(sender_counts) > 0
                else 0
            )
        })

    else:
        email_grouping_results.append({
            "dataset": name,
            "sender_available": False,
            "rows": len(df),
            "rows_with_sender": 0,
            "unique_senders": 0,
            "repeated_senders": 0,
            "rows_in_repeated_sender_groups": 0,
            "largest_sender_group": 0
        })

email_grouping_summary = pd.DataFrame(
    email_grouping_results
)

display(email_grouping_summary)

,dataset,sender_available,rows,rows_with_sender,unique_senders,repeated_senders,rows_in_repeated_sender_groups,largest_sender_group
0,CEAS-08,True,39154,39154,24578,2405,16981,462
1,Enron,False,29767,0,0,0,0,0
2,Nazario,True,1565,1565,1438,69,196,31
3,SpamAssassin,True,5809,5809,2523,462,3748,109


In [45]:
sender_label_results = []

for name in [
    "CEAS-08",
    "Nazario",
    "SpamAssassin"
]:
    df = loaded_datasets[name].copy()

    df["_sender_audit"] = (
        df["sender"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    valid = df[
        df["_sender_audit"] != ""
    ]

    label_counts_per_sender = (
        valid
        .groupby("_sender_audit")["label"]
        .nunique()
    )

    conflicting = label_counts_per_sender[
        label_counts_per_sender > 1
    ]

    sender_label_results.append({
        "dataset": name,
        "unique_senders": valid[
            "_sender_audit"
        ].nunique(),
        "senders_with_multiple_labels":
            len(conflicting),
        "percent_conflicting_senders": round(
            len(conflicting)
            / valid["_sender_audit"].nunique()
            * 100,
            4
        )
    })

sender_label_summary = pd.DataFrame(
    sender_label_results
)

display(sender_label_summary)

,dataset,unique_senders,senders_with_multiple_labels,percent_conflicting_senders
0,CEAS-08,24578,5,0.0203
1,Nazario,1438,0,0.0000
2,SpamAssassin,2523,0,0.0000


## 12. Raw Dataset Reproducibility Inventory

A reproducibility inventory is generated for the raw source files used in
the audit. Each selected source file is identified by its path, file size,
dataset dimensions, and SHA-256 cryptographic hash.

The hashes provide file-level fingerprints that allow the exact audited
inputs to be distinguished from modified or alternative versions of the
same datasets.

No source files are modified during this process.

In [46]:
import hashlib


def sha256_file(path, chunk_size=1024 * 1024):
    sha256 = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()

In [47]:
inventory_records = []

# Main datasets
for name, info in DATASETS.items():

    # ISCX is handled separately because it contains two files.
    if name == "ISCX-URL2016":
        continue

    path = info["path"]
    df = loaded_datasets[name]

    inventory_records.append({
        "dataset": name,
        "modality": info["modality"],
        "file_name": path.name,
        "format": info["format"],
        "rows": len(df),
        "columns": len(df.columns),
        "size_mb": round(
            path.stat().st_size / (1024 ** 2),
            2
        ),
        "sha256": sha256_file(path)
    })


# ISCX files
iscx_info = DATASETS["ISCX-URL2016"]

for path in iscx_info["paths"]:
    df = iscx_datasets[path.name]

    inventory_records.append({
        "dataset": "ISCX-URL2016",
        "modality": "url",
        "file_name": path.name,
        "format": "csv",
        "rows": len(df),
        "columns": len(df.columns),
        "size_mb": round(
            path.stat().st_size / (1024 ** 2),
            2
        ),
        "sha256": sha256_file(path)
    })


dataset_inventory = pd.DataFrame(
    inventory_records
)

display(dataset_inventory)

,dataset,modality,file_name,format,rows,columns,size_mb,sha256
0,CEAS-08,email,CEAS_08.csv,csv,39154,7,64.76,22375e7d5f5a8229dbe987914ee9b3705656c590038662a7df6054629b376074
1,Enron,email,Enron.csv,csv,29767,3,42.76,11f28a5bdd3603f1829681ab6b55071b5f9af781b4e6694e39216b2996a96883
2,Nazario,email,Nazario.csv,csv,1565,7,7.45,b8fbc4158fbdfaff1ed98584c43d72e283c6352d7ade4d457d34c1d79488d184
3,SpamAssassin,email,SpamAssasin.csv,csv,5809,7,14.19,3bfe8f8abff89f69a98456be50413c2fcb20a476141116dd84ee1507b980c00e
4,PhiUSIIL,webpage,PhiUSIIL_Phishing_URL_Dataset.csv,csv,235795,56,54.22,a236549cd369cd80bd478ff8e1779cbf44c58d5c3f79f7a51a1adbed7d06d1c6
5,UCI Phishing Websites,webpage,Training Dataset.arff,arff,11055,31,0.76,3bcf93b1612306537ee2737751f1bac185b2a77886ab376644c3bf0c7f7cb3e6
6,ISCX-URL2016,url,All.csv,csv,36707,80,11.12,39bd96b0c4738daf7be9db7fb9561512afa5599ab34484aa5b7e70c1c55034c5
7,ISCX-URL2016,url,Phishing.csv,csv,15367,80,4.42,99c732baf469f76fe7fa79e28291b3fb685d06b86e307f26ba09fd2ba7d8ae99


## 13. Audit Artifact Export

The principal audit results are exported as structured CSV files to preserve
the evidence produced during this notebook. These artifacts provide a
reproducible record of dataset characteristics and serve as references for
the subsequent cleaning and preprocessing stages.

The exported files contain audit results only. No cleaned or transformed
dataset is produced by this notebook.

In [50]:
[var for var in globals() if "duplic" in var.lower()]

['duplicate_results',
 'duplicate_count',
 'duplicate_percent',
 'duplicate_report',
 'email_duplicate_results',
 'email_content_duplicates']

In [52]:
# Core audit artifacts
dataset_inventory.to_csv(
    AUDIT_DIR / "dataset_inventory.csv",
    index=False
)

label_audit.to_csv(
    AUDIT_DIR / "label_distribution.csv",
    index=False
)

verified_label_semantics.to_csv(
    AUDIT_DIR / "verified_label_semantics.csv",
    index=False
)

missing_report.to_csv(
    AUDIT_DIR / "missing_values.csv",
    index=False
)

duplicate_report.to_csv(
    AUDIT_DIR / "exact_duplicates.csv",
    index=False
)

email_content_statistics.to_csv(
    AUDIT_DIR / "email_content_statistics.csv",
    index=False
)

email_length_distribution.to_csv(
    AUDIT_DIR / "email_length_distribution.csv",
    index=False
)

structured_numeric_audit.to_csv(
    AUDIT_DIR / "structured_numeric_audit.csv",
    index=False
)

uci_native_value_audit.to_csv(
    AUDIT_DIR / "uci_native_value_audit.csv",
    index=False
)

phiusiil_grouping_summary.to_csv(
    AUDIT_DIR / "phiusiil_grouping_summary.csv",
    index=False
)

phiusiil_domain_label_summary.to_csv(
    AUDIT_DIR / "phiusiil_domain_label_summary.csv",
    index=False
)

email_grouping_summary.to_csv(
    AUDIT_DIR / "email_grouping_summary.csv",
    index=False
)

sender_label_summary.to_csv(
    AUDIT_DIR / "email_sender_label_summary.csv",
    index=False
)

print(f"Audit artifacts exported to:\n{AUDIT_DIR}")

Audit artifacts exported to:
/Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/audits
